# Notebook 1 — Prepare Competitor Documents

Downloads the retail split from the C-SEO Bench dataset and saves all competitor
documents (non-target) per query to `competitor_docs.json`.

**Does NOT touch `selected_docs.json`** — safe to run in parallel with Notebook 2.

**Output:** `data/retail/competitor_docs.json`

Structure:
```json
{
  "0": {
    "query_id": 50881,
    "query": "holidaytraditions",
    "target_doc_idx": 5,
    "competitor_docs": [
      {"doc_idx": 0, "doc": "Name: Kids Christmas..."},
      {"doc_idx": 1, "doc": "Name: Graduation..."},
      ...
    ]
  }
}
```

## Instructions
1. Run **Setup**
2. Run **Load Dataset**
3. Run **Save Competitor Docs**
4. Run **Inspect**

## Setup

In [1]:
import json
import os

from datasets import load_dataset

notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, ".."))

data_dir = os.path.join(project_root, "data", "retail")
os.makedirs(data_dir, exist_ok=True)

# Paths
selected_docs_path   = os.path.join(data_dir, "selected_docs.json")    # read-only reference
competitor_docs_path = os.path.join(data_dir, "competitor_docs.json")  # output

print("Setup complete.")
print(f"Project root: {project_root}")
print(f"Output file:  {competitor_docs_path}")

Setup complete.
Project root: /Users/leonardrampf/Library/CloudStorage/OneDrive-Personal/Dokumente/Universität/Nova SBE/Work Project/geo-experiment
Output file:  /Users/leonardrampf/Library/CloudStorage/OneDrive-Personal/Dokumente/Universität/Nova SBE/Work Project/geo-experiment/data/retail/competitor_docs.json


## Load Dataset

Downloads the retail split from HuggingFace — requires internet connection.

In [2]:
print("Loading retail dataset from HuggingFace...")
ds = load_dataset("parameterlab/c-seo-bench", split="retail")
df = ds.to_pandas()

print(f"Loaded {len(df)} rows")
print(f"Columns: {list(df.columns)}")
print(f"Unique queries: {df['query_id'].nunique()}")
print()
print("First row:")
print(df.iloc[0])

Loading retail dataset from HuggingFace...


Loaded 5000 rows
Columns: ['query_id', 'query', 'document']
Unique queries: 500

First row:
query_id                                                50881
query                                       holidaytraditions
document    Name: Graduation Ornament 2021 Guy – Class of ...
Name: 0, dtype: object


## Save Competitor Docs

For each query, saves all documents except the target document.
Target doc index is read from `selected_docs.json`.

**Does NOT modify `selected_docs.json`.**

In [3]:
# Load selected_docs.json to get target_doc_idx per query
with open(selected_docs_path, "r", encoding="utf-8") as f:
    selected_docs = json.load(f)

query_ids = df["query_id"].unique()
competitor_docs = {}

for query_idx, query_id in enumerate(query_ids):
    query_idx_str = str(query_idx)

    if query_idx_str not in selected_docs:
        print(f"[{query_idx}] WARNING: not in selected_docs — skipping")
        continue

    # Get target doc index from selected_docs
    target_doc_idx = int(list(selected_docs[query_idx_str].keys())[0])

    # Get all docs for this query
    hits = df[df["query_id"] == query_id].reset_index(drop=True)
    query_text = hits["query"].iloc[0]

    # Save all competitor docs (all except target)
    competitors = []
    for doc_idx, row in hits.iterrows():
        if doc_idx == target_doc_idx:
            continue  # skip target doc
        competitors.append({
            "doc_idx": doc_idx,
            "doc": row["document"],
        })

    competitor_docs[query_idx_str] = {
        "query_id":        int(query_id),
        "query":           query_text,
        "target_doc_idx":  target_doc_idx,
        "competitor_docs": competitors,
    }

# Save to file — does NOT touch selected_docs.json
with open(competitor_docs_path, "w", encoding="utf-8") as f:
    json.dump(competitor_docs, f, indent=4, ensure_ascii=False)

print(f"Saved competitor_docs.json — {len(competitor_docs)} queries")
print(f"Example: query 0 has {len(competitor_docs['0']['competitor_docs'])} competitor docs")

Saved competitor_docs.json — 500 queries
Example: query 0 has 9 competitor docs


## Inspect

In [4]:
with open(competitor_docs_path, "r", encoding="utf-8") as f:
    competitor_docs = json.load(f)

INSPECT_QUERY_IDX = "0"
example = competitor_docs[INSPECT_QUERY_IDX]

print(f"Query idx:       {INSPECT_QUERY_IDX}")
print(f"Query ID:        {example['query_id']}")
print(f"Query:           {example['query']}")
print(f"Target doc idx:  {example['target_doc_idx']}")
print(f"Competitor docs: {len(example['competitor_docs'])}")
print()
print("First competitor doc (first 200 chars):")
print(example['competitor_docs'][0]['doc'][:200])

Query idx:       0
Query ID:        50881
Query:           holidaytraditions
Target doc idx:  5
Competitor docs: 9

First competitor doc (first 200 chars):
Name: Graduation Ornament 2021 Guy – Class of 2021 Ornament – Personalized Christmas Ornaments – School, Teacher Ornaments – Unique Graduation Gift for Him – Polyresin Graduation Decorations 2021
Desc
